In [ ]:
import numpy as np
import os
import pandas as pd 
import sys
import subprocess
from matplotlib import pyplot as plt
from segmentation import segment_cough, compute_SNR
from feature_class import *
from tqdm import tqdm_notebook

# final model should be a version that takes this model + the cough classification model 

In [ ]:
data = pd.read_csv("../coughvidDataset/metadata_compiled.csv")
from tqdm import tqdm_notebook

In [ ]:
coughTypeColumns = ["cough_type_1", "cough_type_2", "cough_type_3", "cough_type_4"]
expertsFilter = data[coughTypeColumns].apply(lambda x: x.dropna().nunique() == 1, axis=1)
data = data.loc[expertsFilter]
data["cough_type"] = data[coughTypeColumns].bfill(axis=1).iloc[:, 0]
data = data[["uuid", "cough_type"]]
data = data.drop(data[data["cough_type"] == "unknown"].index)

In [ ]:
data.head(5)

In [ ]:
feats = features([5]) # random number for FREQ_CUTS because the paper said that this wasn't really useful

In [ ]:
finalDataset = []

for file in tqdm_notebook(data["uuid"]):
    for i in os.listdir("../coughvidDataset/"):
        i=i.split(".")
        if i[0] != file or i[1] != "webm":
            continue

        path = path = rf"C:\cac2026\coughvidDataset\{file}.webm"
        result = subprocess.run(
            ["ffmpeg", "-i", path, "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=True
        )

        signal = np.frombuffer(result.stdout, dtype=np.float32)
        fs = 16000

        # print("sig", signal)
        # print(file)
        cough_segments, cough_mask = segment_cough(signal, fs, cough_padding=0)
        # plt.plot(signal)
        # plt.plot(cough_mask)
        # plt.title("Segmentation Output")
        # fig, axs = plt.subplots(len(cough_segments),1, figsize=(7,10))
        # for i in range(0,len(cough_segments)):
        #     axs[i].plot(cough_segments[i])
        #     axs[i].set_title("Cough segment " + str(i))
        snr = compute_SNR(signal,fs)

        featureFunctions = [
            feats.EEPD,
            feats.PRE,
            feats.ZCR,
            feats.RMSP,
            feats.DF,
            feats.spectral_features,
            feats.SF_SSTD,
            feats.SSL_SD,
            feats.MFCC,
            feats.CF,
            feats.LGTH,
            feats.MFCC_DELTA,
            feats.MFCC_DELTA2,
            feats.spectralEntropy
        ]

        featPacket = [fs, signal]
        featureArray = np.concatenate([
            np.asarray(function(featPacket)[0]).flatten() for function in featureFunctions
        ])
        featureArray = np.append(featureArray, data[data["uuid"] == file].iloc[0]["cough_type"]) # wet is 1, dry is 0
        finalDataset.append(featureArray)

In [ ]:
X = [i[:-1] for i in finalDataset]
y = [int(i[-1] == "wet") for i in finalDataset]

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

XTrain, XTest, yTrain, yTest = train_test_split(
    np.array(X), np.array(y), test_size=0.2, random_state=42, stratify=y
)

# from sklearn.utils import resample

# XDry = XTrain[yTrain == 0]
# yDry = yTrain[yTrain == 0]

# XWet = XTrain[yTrain == 1]
# yWet = yTrain[yTrain == 1]

# XWetOversampled, yWetOversampled = resample(
#     XWet,
#     yWet,
#     replace=True,
#     n_samples=1500,
#     random_state=42
# )

# XTrain = np.vstack((XDry, XWetOversampled))
# yTrain = np.concatenate((yDry, yWetOversampled))
# for weight in [2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8]:
#     model = XGBClassifier(
#         colsample_bytree=0.5732957584868299, 
#         gamma=3.2697672617295446,
#         learning_rate=0.22855810864114873,
#         max_delta_step=10.0,
#         max_depth=40,
#         min_child_weight=19.0,
#         n_estimators=50,
    #     reg_alpha=52.0,
    #     reg_lambda=0.11594407730658135,
    #     scale_pos_weight=weight
    # )
    # model.fit(XTrain, yTrain)

    # yPred = model.predict(XTest)

    # print("\nWeight:", weight)
    # print(classification_report(
    #     yTest, yPred,
    #     target_names=["dry", "wet"]
    # ))
model = XGBClassifier(
    colsample_bytree=0.5732957584868299, 
    gamma=3.2697672617295446,
    learning_rate=0.22855810864114873,
    max_delta_step=10.0,
    max_depth=13,
    min_child_weight=19.0,
    n_estimators=500,
    reg_alpha=52.0,
    reg_lambda=0.11594407730658135,
    scale_pos_weight=2.3
)
model.fit(XTrain, yTrain)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

yPred = model.predict(XTest)

print("Accuracy:", accuracy_score(yTest, yPred))
print(classification_report(yTest, yPred, target_names=["dry", "wet"]))

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier 

model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_depth=10,
    min_samples_leaf=5,
    l2_regularization=51.0,
    random_state=42
)

model.fit(XTrain, yTrain)
yPred = model.predict(XTest)
print("Accuracy:", accuracy_score(yTest, yPred))
print(classification_report(
    yTest, yPred,
    target_names=["dry", "wet"]
))

In [ ]:
import joblib

joblib.dump(model, "histGradientBoostingModel.joblib")

In [ ]:
model = joblib.load("histGradientBoostingModel.joblib")
model.predict(XTest)